# Per-object Kerchunk Reference Generation

Generates one parquet reference directory per new or changed source object.
Ledger is committed only when all generation tasks succeed.

In [ ]:
%run ./01.1-S3_inventory_diff

In [ ]:
import sys
from pathlib import Path

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pipeline.generate_parquet import generate_references_in_parallel, save_ledger_after_success

import virtualizarr
import obstore
import obspec_utils
print('Imports OK')

In [ ]:
reference_generation = generate_references_in_parallel(
    kp=kp,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    inventory_diff=inventory_diff,
    current_objects=inventory_objects,
)

print('Reference generation summary:', reference_generation['summary'])
print('Generated sample:', [r['key'] for r in reference_generation['results'] if r['status'] == 'generated'][:10])
print('Failures:', reference_generation['failures'][:5])

## Commit Ledger Update

Commit is blocked when any reference generation error is present.

In [ ]:
if reference_generation['summary']['failed'] == 0:
    save_ledger_after_success(
        ledger_path=kp['output']['ledger_path'],
        next_ledger=pending_ledger,
        generation_summary=reference_generation['summary'],
    )
    print('Ledger committed:', kp['output']['ledger_path'])
else:
    print('Ledger NOT committed due to generation failures.')